## No memory

In [4]:
%run langchain_common.py

# make sure to start mlflow server before running this notebook
# example(run this from terminal): mlflow ui --port 5000 

mlflow.set_experiment("langchain_agents_memory_demo")

agent = create_agent(llm_noreason)

Trace(trace_id=tr-10f191804d2f0e149a8c7e22dc194afa)

In [5]:
question = HumanMessage(content="Hello my name is Ali and my favourite colour is green")

response = agent.invoke(
    {"messages": [question]} 
)
pp(response["messages"][-1].content)

"Hello Ali! It's nice to meet you. Green is a wonderful choice—it often symbolizes nature, growth, and calmness. 🌿\n\nIs there anything specific you'd like to chat about or need help with today?"


In [6]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]} 
)

pp(response["messages"][-1].content)

"I don't know your favorite color yet! As an AI, I don't have access to your personal preferences or memories unless you've told me in this specific conversation.\n\nHowever, if you tell me, I'll remember it for the rest of our chat. Do you have a favorite?"


## Memory

In [7]:
# Checkpoints persist conversation state between turns so the agent can remember prior context.
# InMemorySaver is an in-process checkpoint backend: fast and simple for demos, but non-persistent across kernel restarts.

# Clarification:
# `messages` in agent.invoke(...) is only the per-call input payload (what you send this turn).
# Checkpoint memory is different: it stores and reloads the full conversation state across turns.
# That persisted state is keyed by `thread_id`, so memory works only when the same thread_id is reused.
# In short: message context is request-scoped; checkpoint memory is thread-scoped and persistent (in-process here).

from langgraph.checkpoint.memory import InMemorySaver  

agent = create_agent(
    llm_noreason,
    checkpointer=InMemorySaver(),  
)

Trace(trace_id=tr-d9b7e0b7395c5e72b3200ed2664bddd3)

In [8]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Hello my name is Ali and my favourite colour is green")
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [question]},
    config,  
)

In [9]:
pp(response["messages"][-1].content)

"Hello Ali! It's nice to meet you. Green is a wonderful choice—it often symbolizes nature, growth, and calmness. 🌿\n\nIs there anything specific you'd like to chat about or need help with today?"


In [10]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]},
    config,  
)

resp = response["messages"][-1].content
pp(resp)

'Your favourite colour is **green**! 🌿\n\nYou mentioned that earlier in our conversation. Is there a specific shade of green you like the most, like forest green, lime, or mint?'


[Trace(trace_id=tr-622494d2bd93995ac05a571a618cec3d), Trace(trace_id=tr-f41fd4f61763a67ab04d8c4f7b03702e)]

In [11]:
# Demonstrate storing custom data in memory for the same thread
user_data = {
    "name": "Ali",
    "favorite_color": "green",
    "city": "Lahore",
    "loyalty_tier": "gold"
}

# Save custom data into the thread memory
save_message = HumanMessage(
    content=(
        "Store this custom data in memory and use it for future answers:\n"
        f"{user_data}"
    )
)
agent.invoke({"messages": [save_message]}, config)

# Ask a follow-up question that requires the stored custom data
question = HumanMessage(
    content="From my saved profile, what city do I live in and what is my loyalty tier?"
)
response = agent.invoke({"messages": [question]}, config)
pp(response["messages"][-1].content)

"Based on your saved profile, you live in **Lahore** and your loyalty tier is **Gold**. 🏆🌆\n\nIs there anything specific you'd like to explore regarding your Gold tier benefits or things to do in Lahore?"


[Trace(trace_id=tr-435ffd4ffd56c13278e246d66faca9c4), Trace(trace_id=tr-2392a71f808c7879f9a9a5c0988356c5)]

In [12]:
# Inspect recent messages stored in memory for this thread
state = agent.get_state(config)
pp(state.values["messages"][-4:])

[
    HumanMessage(content="Store this custom data in memory and use it for future answers:\n{'name': 'Ali', 'favorite_color': 'green', 'city': 'Lahore', 'loyalty_tier': 'gold'}", additional_kwargs={}, response_metadata={}, id='62ad0b34-484b-475c-8e19-8e8fac96f2a4'),
    AIMessage(content="Got it, Ali! I've stored your details in my context for this conversation:\n\n*   **Name:** Ali\n*   **Favorite Color:** Green 🌿\n*   **City:** Lahore\n*   **Loyalty Tier:** Gold 🏆\n\nI'll keep these in mind for our future answers. Since you're a **Gold** member from **Lahore**, is there anything specific you'd like to know about local events, green-themed recommendations, or perhaps some exclusive perks associated with your tier?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 180, 'total_tokens': 289, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen35-122b-a10b', 's

In [13]:
import uuid

def new_conversation_id() -> str:
    return str(uuid.uuid4())
  
def make_thread_config(user_id: str) -> dict:
    conversation_id = new_conversation_id()
    return {"configurable": {"thread_id": f"user-{user_id}:conv-{conversation_id}"}}

def ask(thread_config, text):
    result = agent.invoke({"messages": [HumanMessage(content=text)]}, thread_config)
    result =  result["messages"][-1].content
    pp(thread_config["configurable"]["thread_id"] + ":" + result)

In [14]:

# Scenario 1: Thread isolation (different users/sessions)
config_a = make_thread_config("27100001")
config_b = make_thread_config("27100002")

ask(config_a, "My preferred database is PostgreSQL.")
ask(config_b, "My preferred database is MongoDB.")

ask(config_a, 'What database do I prefer?')
ask(config_b, 'What database do I prefer?')

'user-27100001:conv-ad12f10c-416b-4450-a4b7-f2ecbb6f016e:Great! PostgreSQL is an excellent choice known for its reliability, advanced features (like JSONB support, full-text search, and robust concurrency control), and strong standards compliance.\n\nHow can I help you with PostgreSQL? For example:\n- Designing a schema or optimizing queries\n- Setting up replication, backups, or security\n- Debugging performance issues or slow queries\n- Migrating from another database\n- Exploring specific features (e.g., partitioning, extensions like PostGIS)\n- Writing complex SQL (CTEs, window functions, stored procedures)\n\nLet me know your goal or challenge! 🐘'
'user-27100002:conv-cc9975fe-5c72-4112-8569-c63e780be46c:Great! MongoDB is a powerful, flexible NoSQL database known for its document-oriented structure, scalability, and ease of use. How can I help you? For example:\n\n- Designing a schema or data model?\n- Writing queries (CRUD operations, aggregations, etc.)?\n- Setting up replication

[Trace(trace_id=tr-12beb0d281c1ea17b4cc42662c06c35f), Trace(trace_id=tr-db5da335ab896697e9abd698c79fc1de), Trace(trace_id=tr-1a17bfea4da5768b3f76b647529e3688), Trace(trace_id=tr-0d5e2c352cbe87ddf7d8b27894951cd1)]

In [15]:
# Scenario 2: Preference update within the same thread
config_update = make_thread_config("27100003")

ask(config_update, "My deployment region is East US.")
ask(config_update, "Actually, update that: my deployment region is West Europe.")
ask(config_update, 'Which deployment region should you use now?')

'user-27100003:conv-09e2d100-446b-4041-b365-e8e2eb04a1a2:Got it! Your deployment region is **East US**. This is a popular Azure region located in Virginia, USA, known for low latency to the US East Coast and strong compliance with US data residency requirements.\n\nHow can I help? For example:\n- Need guidance on deploying resources (VMs, databases, etc.) in East US?\n- Looking for region-specific pricing or service availability?\n- Troubleshooting latency or connectivity issues?\n- Planning a multi-region architecture with East US as primary?\n\nLet me know your goal! 😊'
"user-27100003:conv-09e2d100-446b-4041-b365-e8e2eb04a1a2:Noted! I've updated your deployment region to **West Europe**.\n\nThis region is located in the **Netherlands** and is a key hub for organizations targeting the European market, offering:\n*   **Low latency** for users across Western and Central Europe.\n*   **GDPR compliance** and data residency within the EU.\n*   Strong connectivity to major European internet

Trace(trace_id=tr-f26c583e85d6565b5bf83c6aa72b38e0)

In [16]:

# Scenario 3: Multi-step task memory
config_task = make_thread_config("27100004")

ask(config_task, "I'm building a chatbot MVP. Deadline is Friday. Budget is $2,000.")
pp(f"Scenario 3A: Recall previous constraints:")
ask(config_task, 'Summarize my constraints in one sentence.')
pp(f"Scenario 3B: Suggest next steps:")
ask(config_task, 'Given those constraints, suggest 3 practical next steps.')

task_state = agent.get_state(config_task)
pp("\nRecent messages in Scenario 3 thread:")
pp(task_state.values["messages"][-4:])

'user-27100004:conv-c293e715-4fa9-44cb-a965-bdcba27f802f:That\'s a tight but achievable goal! With a **$2,000 budget** and a **Friday deadline**, you need to prioritize speed, simplicity, and leveraging existing tools rather than building from scratch. Here\'s a streamlined plan:\n\n---\n\n### **1. Define Core Scope (Critical!)**\n- **Must-haves only**: 1–2 key use cases (e.g., FAQ answers, lead capture, or basic troubleshooting).\n- **No custom NLP training**: Use pre-trained models (e.g., Dialogflow, Rasa, or LLM APIs).\n- **No complex integrations**: Stick to 1–2 channels (e.g., web chat + WhatsApp/Telegram).\n- **Manual fallback**: Human handoff for edge cases (saves dev time).\n\n---\n\n### **2. Tech Stack Recommendations**\n| Component          | Tool Options (Free/Low-Cost)       | Why?                                  |\n|--------------------|------------------------------------|---------------------------------------|\n| **Chatbot Engine** | Dialogflow ES (free tier), Botpress

[Trace(trace_id=tr-e16b0a7e1b82540ea6c02879afff5d10), Trace(trace_id=tr-2cb8f2d6fee6e9104948c0ff262d7b53), Trace(trace_id=tr-99d63215f71f03f3665173afd0188bd4)]